# Person 4 — Cohort retention + Monitoring

**Q4.** Group customers by the month of their first order (cohort). For each 1996 cohort,
what share of customers placed another order within the next three months?

Definitions:
- **cohort** = month of the customer's first order (`MIN(o_orderdate)`)
- **returned** = the customer's second order date ≤ first order date + 3 months (inclusive)

Requires Gold `customer_marketing` (run `repeat_purchase_rate_analysis` or `run_pipeline` first).

In [ ]:
from pyspark.sql import SparkSession

from src.analysis.cohort_retention import calculate_cohort_retention

spark = SparkSession.getActiveSession()

cohorts = calculate_cohort_retention(spark)
display(cohorts)

In [ ]:
from pyspark.sql import functions as F

total = cohorts.agg(
    F.sum("cohort_customers").alias("customers"),
    F.sum("returned_customers").alias("returned"),
).collect()[0]
print(f"All 1996 cohorts: {total.returned}/{total.customers} = {total.returned / total.customers:.2%}")

best = cohorts.orderBy(F.col("retention_rate").desc()).first()
worst = cohorts.orderBy("retention_rate").first()
print(f"Best cohort:  {best.cohort} ({best.retention_rate:.2%})")
print(f"Worst cohort: {worst.cohort} ({worst.retention_rate:.2%})")

## Visualization — 3-month retention by 1996 cohort

Chart type **Bar** (or Combo): X = `cohort`, Y = `retention_rate`; optionally `cohort_customers` on a second axis.

In [ ]:
display(cohorts.select("cohort", "cohort_customers", "returned_customers", "retention_rate"))

## Monitoring

- `monitoring_activation_rate_by_segment` — cumulative activation rate per segment at each month end
- `monitoring_new_customers_monthly` — new customers (by first order date) per month

Only complete months are included. Alert rules (see `src/monitoring/marketing_monitoring.py`):
- activation rate of a segment drops by more than **1 pp** month over month
- new customers in a month fall below **50%** of the trailing 3-month average (only when that average ≥ 10)

In [ ]:
from src.monitoring.marketing_monitoring import active_alerts, build_monitoring

activation, new_customers = build_monitoring(spark)

### Activation rate by segment over time

Chart type **Line**: X = `month`, Y = `activation_rate`, group by `market_segment`.

In [ ]:
display(activation.select("month", "market_segment", "activation_rate"))

### New customers per month

Chart type **Line** or **Bar**: X = `month`, Y = `new_customers`.

In [ ]:
display(new_customers)

### Alerts

In [ ]:
display(active_alerts(spark))